# 02 - Entrenamiento y evaluacion: SimplifiedPointNet sobre ModelNet10

Pensado para **Google Colab con GPU T4**. Requiere haber ejecutado antes
(o ejecutar aqui mismo) la descarga/preprocesado de `notebooks/01_data_exploration.ipynb` -
este notebook reutiliza la misma cache en `data/`.

Contenido:
1. Setup (clonar repo, dependencias)
2. Entrenamiento (`src/train.py`)
3. Curvas de loss/accuracy por epoca
4. Evaluacion en test: accuracy, classification report, matriz de confusion
5. Visualizacion de predicciones concretas

## 0. Setup

In [ ]:
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    if not os.path.isdir("3d_classifier"):
        !git clone https://github.com/marsaliborra/3d_classifier.git
    %cd 3d_classifier
    !git pull
    !pip install -q trimesh
    print("Ejecutando en Colab.")
else:
    print("Ejecutando localmente (asumo que ya estoy en la raiz del repo).")

In [ ]:
import torch

print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("AVISO: sin GPU, el entrenamiento sera lento. Runtime > Change runtime type > T4 GPU.")

## 1. Entrenamiento

`train()` descarga/cachea el dataset si hace falta (reutiliza la cache si
ya se genero en el notebook 01) y entrena `SimplifiedPointNet` con
`CrossEntropyLoss` + `Adam`. Guarda el mejor checkpoint (`best_model.pt`,
segun accuracy de test) y el historial de metricas por epoca
(`history.json`) en `outputs/checkpoints/`.

Con 30 epocas, batch 32 y una T4, deberia tardar del orden de varios
minutos (el tiempo exacto depende de cuanta cola haya en la GPU compartida
de Colab).

In [ ]:
from src.train import train

model, history = train()

## 2. Curvas de aprendizaje

Loss y accuracy de train vs test por epoca. Si train sigue mejorando
mientras test se estanca o empeora, es la senal de overfitting - con un
dataset y modelo tan pequenos, vale la pena revisarlo antes de fiarse solo
de la accuracy final.

In [ ]:
from pathlib import Path
from src.evaluate import plot_training_curves

plot_training_curves(history, Path("outputs/figures/training_curves.png"))

## 3. Evaluacion en test

Carga el mejor checkpoint guardado durante el entrenamiento y calcula:
- accuracy global en el test set
- un `classification_report` de sklearn (precision/recall/F1 por clase -
  util para ver si el modelo falla de forma pareja o se concentra en
  confundir un par de clases geometricamente parecidas, p.ej. desk/table)
- la matriz de confusion (normalizada por fila, con seaborn)

In [ ]:
from src.evaluate import (
    CHECKPOINT_DIR,
    FIGURES_DIR,
    load_trained_model,
    plot_confusion_matrix,
    plot_example_predictions,
    predict_all,
)
from src.data import download_modelnet10, get_dataset_arrays
from src.train import N_POINTS
from sklearn.metrics import classification_report
from src.data import CLASSES

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset_dir = download_modelnet10(root="data")
test_points, test_labels = get_dataset_arrays(dataset_dir, "test", n_points=N_POINTS, cache_dir="data")

best_model = load_trained_model(CHECKPOINT_DIR / "best_model.pt", device)
predictions = predict_all(best_model, test_points, device)

accuracy = (predictions == test_labels).mean()
print(f"Accuracy en test: {accuracy:.4f} ({(predictions == test_labels).sum()}/{len(test_labels)})\n")
print(classification_report(test_labels, predictions, target_names=CLASSES))

In [ ]:
plot_confusion_matrix(test_labels, predictions, FIGURES_DIR / "confusion_matrix.png")

## 4. Ejemplos de predicciones

4 nubes de puntos aleatorias del test set con su etiqueta real y la
prediccion del modelo (verde = acierto, rojo = error).

In [ ]:
plot_example_predictions(test_points, test_labels, predictions, FIGURES_DIR / "example_predictions.png")

## Siguiente paso

Con la accuracy, la matriz de confusion y los ejemplos ya generados,
toca volcar los resultados reales al `README.md` del repo (sustituyendo
los `TODO`): que funciono, que confunde el modelo y por que, y que se
haria distinto con mas tiempo.